**Seleccionar un cliente para la muestra**

In [0]:
%sql
SELECT * FROM andina_market.gold.dim_customer WHERE CustomerID = 1;

CustomerID,FullName,Email,Phone,City,Country,Segment,SignupDate,CreatedAt,UpdatedAt,_processed_at,__START_AT,__END_AT,_ingested_at
1,Hernán Guevara Garica,null,+56333101338,Lima,null,Regular,2023-03-20,2023-03-20T21:01:04.659Z,2023-03-20T21:01:04.659Z,2026-08-31T15:47:03.345Z,2023-03-20T21:01:04.659Z,2026-08-31T18:02:18.412Z,null
1,Hernán Guevara Garica,hernan@email.com,+56333101338,Lima,Peru,VIP,2023-03-20,2023-03-20T21:01:04.659Z,2026-08-31T18:02:18.412Z,2026-08-31T18:02:18.412Z,2026-08-31T18:02:18.412Z,null,2026-08-31T18:02:18.412Z


Simular que el cliente pasa a ser cliente VIP con un `UpdatedAt` más reciente (por ejemplo, fecha de hoy):

In [0]:
%sql
INSERT INTO andina_market.silver.customers 
VALUES (
  1, 
  'Hernán Guevara Garica', 
  'hernan@email.com', 
  '+56333101338', 
  'Lima', 
  'Peru', 
  'VIP', 
  '2023-03-20', 
  '2023-03-20T21:01:04.659+00:00', 
  CURRENT_TIMESTAMP(), 
  CURRENT_TIMESTAMP(), 
  CURRENT_TIMESTAMP()
);

num_affected_rows,num_inserted_rows
1,1


**Correr Pipeline ETL de nuevo**

**Ver resultados**

Podemos observar que se mantienen ambos registros. La columna `__END_AT` indica cuando se realizó el cambio.  

In [0]:
%sql
SELECT 
    CustomerID,
    FullName,
    Email,
    Phone,
    City,
    Country,
    Segment,
    UpdatedAt,
    __START_AT,
    __END_AT
FROM 
    andina_market.gold.dim_customer 
WHERE 
    CustomerID = 1;

CustomerID,FullName,Email,Phone,City,Country,Segment,UpdatedAt,__START_AT,__END_AT
1,Hernán Guevara Garica,null,+56333101338,Lima,null,Regular,2023-03-20T21:01:04.659Z,2023-03-20T21:01:04.659Z,2026-08-31T18:02:18.412Z
1,Hernán Guevara Garica,hernan@email.com,+56333101338,Lima,Peru,VIP,2026-08-31T18:02:18.412Z,2026-08-31T18:02:18.412Z,null


**Identificar clientes que cambiaron de segmento**

In [0]:
%sql
-- Query para identificar cambios de segmento
WITH customer_changes AS (
    SELECT 
        CustomerID,
        FullName,
        Segment AS SegmentAnterior,
        LEAD(Segment) OVER (PARTITION BY CustomerID ORDER BY __START_AT) AS SegmentNuevo,
        __START_AT AS FechaInicio,
        __END_AT AS FechaFin,
        LEAD(__START_AT) OVER (PARTITION BY CustomerID ORDER BY __START_AT) AS FechaCambio
    FROM 
        andina_market.gold.dim_customer
)
SELECT 
    CustomerID,
    FullName,
    SegmentAnterior,
    SegmentNuevo,
    FechaInicio,
    FechaFin,
    FechaCambio
FROM 
    customer_changes
WHERE 
    SegmentNuevo IS NOT NULL 
    AND SegmentAnterior != SegmentNuevo
ORDER BY 
    CustomerID, FechaInicio;

CustomerID,FullName,SegmentAnterior,SegmentNuevo,FechaInicio,FechaFin,FechaCambio
1,Hernán Guevara Garica,Regular,VIP,2023-03-20T21:01:04.659Z,2026-08-31T18:02:18.412Z,2026-08-31T18:02:18.412Z
